<a href="https://colab.research.google.com/github/shravya1998/indiaaihackathon_cdsco/blob/main/CDSCO_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pypdf faiss-cpu ollama gradio

In [ ]:
!pip install colab-xterm
%load_ext colabxterm

In [ ]:
!pip install zstd

In [ ]:
!sudo apt-get update

In [ ]:
!sudo apt-get install -y pciutils

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
%xterm

In [ ]:
import re
import difflib
import hashlib
from dataclasses import dataclass
from pathlib import Path
from typing import List, Dict, Any, Tuple

import numpy as np
import faiss
import spacy
import ollama
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer


# =========================
# CONFIG
# =========================
OLLAMA_MODEL = "mistral"
EMBED_MODEL_NAME = "BAAI/bge-small-en"
SPACY_MODEL = "en_core_web_lg"

GUIDELINE_PDFS = [
    "ICMR_National_Ethical_Guidelines.pdf",
    "DPDPACT.pdf",
]


# =========================
# LOAD MODELS ONCE
# =========================
print("Loading models...")

try:
    nlp = spacy.load(SPACY_MODEL)
except OSError:
    print(f"{SPACY_MODEL} not found. Falling back to en_core_web_sm.")
    nlp = spacy.load("en_core_web_sm")

embed_model = SentenceTransformer(EMBED_MODEL_NAME)


# =========================
# RULES
# =========================
RULES = {
    "EMAIL": re.compile(r"\b[\w\.-]+@[\w\.-]+\.\w+\b"),
    "PHONE": re.compile(r"\b(?:\+91[- ]?)?[6-9]\d{9}\b"),
    "PAN": re.compile(r"\b[A-Z]{5}[0-9]{4}[A-Z]\b"),
    "AADHAAR": re.compile(r"\b\d{4}\s?\d{4}\s?\d{4}\b"),
    "DATE": re.compile(r"\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b"),
}


@dataclass
class Span:
    start: int
    end: int
    label: str
    source: str


# =========================
# PDF TEXT EXTRACTION
# =========================
def extract_text_from_pdf(file_path: str) -> str:
    file_path = str(file_path)

    if not Path(file_path).exists():
        raise FileNotFoundError(f"PDF not found: {file_path}")

    reader = PdfReader(file_path)
    text_parts = []

    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text_parts.append(page_text)

    return "\n".join(text_parts)


# =========================
# GUIDELINE RAG INDEX
# =========================
def chunk_text(text: str, size: int = 700, overlap: int = 100) -> List[str]:
    chunks = []
    start = 0

    while start < len(text):
        end = start + size
        chunks.append(text[start:end])
        start += size - overlap

    return chunks


def build_guideline_index(pdf_paths: List[str]) -> Tuple[Any, List[str]]:
    print("Building guideline index...")

    chunks = []

    for pdf_path in pdf_paths:
        path = Path(pdf_path)

        if not path.exists():
            print(f"Skipping missing guideline file: {pdf_path}")
            continue

        try:
            text = extract_text_from_pdf(str(path))
            chunks.extend(chunk_text(text))
        except Exception as e:
            print(f"Could not read guideline file {pdf_path}: {e}")

    if not chunks:
        print("No guideline chunks found. RAG retrieval will use fallback text.")
        return None, []

    embeddings = embed_model.encode(chunks, normalize_embeddings=True)
    embeddings = np.asarray(embeddings, dtype="float32")

    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)

    print("Guideline index ready!")
    return index, chunks


guideline_index, guideline_chunks = build_guideline_index(GUIDELINE_PDFS)


def retrieve_guidelines(query: str, k: int = 3) -> str:
    if guideline_index is None or not guideline_chunks:
        return (
            "Fallback guideline: Remove direct identifiers, pseudonymize personal data, "
            "generalize quasi-identifiers, minimize PHI/PII exposure, and report residual risk."
        )

    q_emb = embed_model.encode([query], normalize_embeddings=True)
    q_emb = np.asarray(q_emb, dtype="float32")

    _, idx = guideline_index.search(q_emb, k)

    selected = []
    for i in idx[0]:
        if 0 <= i < len(guideline_chunks):
            selected.append(guideline_chunks[i])

    return "\n".join(selected)


# =========================
# PII / PHI DETECTION
# =========================
def rule_detect(text: str) -> List[Span]:
    spans = []

    for label, pattern in RULES.items():
        for match in pattern.finditer(text):
            spans.append(Span(match.start(), match.end(), label, "rule"))

    return spans


def nlp_detect(text: str) -> List[Span]:
    doc = nlp(text)
    spans = []

    for ent in doc.ents:
        if ent.label_ == "PERSON":
            spans.append(Span(ent.start_char, ent.end_char, "PERSON", "nlp"))
        elif ent.label_ in {"GPE", "LOC"}:
            spans.append(Span(ent.start_char, ent.end_char, "LOCATION", "nlp"))
        elif ent.label_ == "ORG":
            spans.append(Span(ent.start_char, ent.end_char, "ORG", "nlp"))
        elif ent.label_ == "DATE":
            spans.append(Span(ent.start_char, ent.end_char, "DATE", "nlp"))

    return spans


def merge_spans(spans: List[Span]) -> List[Span]:
    if not spans:
        return []

    spans = sorted(spans, key=lambda s: (s.start, -(s.end - s.start)))
    merged = []

    for span in spans:
        if not merged:
            merged.append(span)
            continue

        last = merged[-1]

        if span.start < last.end:
            if span.end > last.end:
                merged[-1] = Span(
                    start=last.start,
                    end=span.end,
                    label=last.label,
                    source=last.source + "+" + span.source,
                )
        else:
            merged.append(span)

    return merged


# =========================
# PSEUDONYMIZATION
# =========================
def anonymize(text: str, spans: List[Span]) -> Tuple[str, Dict[str, str]]:
    out = []
    last = 0
    token_map = {}
    counters = {}

    for span in spans:
        out.append(text[last:span.start])
        original = text[span.start:span.end]

        if original not in token_map:
            counters[span.label] = counters.get(span.label, 0) + 1
            token_map[original] = f"[{span.label}_{counters[span.label]}]"

        out.append(token_map[original])
        last = span.end

    out.append(text[last:])
    return "".join(out), token_map


# =========================
# LLM HELPERS
# =========================
def call_llm(prompt: str, system: str = "You are a strict regulatory AI assistant.") -> str:
    response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": prompt},
        ],
        options={"temperature": 0},
    )

    return response["message"]["content"]


def refine_with_llm(text: str) -> str:
    rules = retrieve_guidelines("PII PHI anonymization DPDP CDSCO ICMR privacy rules")

    prompt = f"""
You are a strict data anonymization and compliance system.

REGULATORY GUIDELINES:
{rules}

TASK:
1. Remove remaining PII/PHI.
2. Keep existing pseudonym tokens such as [PERSON_1], [PHONE_1].
3. Generalize quasi-identifiers.
4. Prevent re-identification.
5. Ensure irreversible anonymization.

Rules:
- Never keep real names.
- Never keep phone/email/PAN/Aadhaar.
- Age should be generalized to range.
- Location should be generalized to broad region only.
- Return only final anonymized text.

INPUT:
{text}
"""

    return call_llm(prompt, "You are a strict anonymization engine.")


# =========================
# COMPLIANCE CHECK
# =========================
def compliance_check(text: str) -> Dict[str, Any]:
    issues = []

    checks = {
        "Phone present": r"\b(?:\+91[- ]?)?[6-9]\d{9}\b",
        "Email present": r"\b[\w\.-]+@[\w\.-]+\.\w+\b",
        "PAN present": r"\b[A-Z]{5}[0-9]{4}[A-Z]\b",
        "Aadhaar present": r"\b\d{4}\s?\d{4}\s?\d{4}\b",
    }

    for issue, pattern in checks.items():
        if re.search(pattern, text):
            issues.append(issue)

    return {
        "status": "PASS" if not issues else "FAIL",
        "issues": issues,
    }


# =========================
# DOCUMENT CLASSIFICATION
# =========================
def classify_doc_type(text: str) -> str:
    t = text.lower()

    if any(word in t for word in ["serious adverse event", "adverse event", "sae", "causality"]):
        return "SAE"

    if any(word in t for word in ["meeting", "minutes", "agenda", "committee"]):
        return "MEETING"

    return "APPLICATION"


# =========================
# SUMMARIZATION
# =========================
def summarize(text: str, doc_type: str) -> str:
    if doc_type == "SAE":
        task = """
Extract:
- Event
- Onset date
- Severity
- Outcome
- Causality
- Missing SAE fields
"""

    elif doc_type == "MEETING":
        task = """
Extract:
- Decisions
- Action items
- Owners
- Due dates
- Next steps
"""

    else:
        task = """
Extract:
- Applicant
- Drug
- Indication
- Trial phase
- CMC status
- Clinical data status
- Labeling status
- Missing application sections
"""

    prompt = f"""
You are a CDSCO regulatory reviewer.

TASK:
{task}

TEXT:
{text}

Return structured YAML only.
"""

    return call_llm(prompt, "You are a strict CDSCO summarization engine.")


# =========================
# REGULATORY COMPLIANCE
# =========================
def regulatory_compliance(summary: str, doc_type: str) -> str:
    rules = retrieve_guidelines(f"{doc_type} drug approval checklist required fields CDSCO ICMR")

    prompt = f"""
Check whether the summary follows regulatory expectations.

GUIDELINES:
{rules}

DOCUMENT TYPE:
{doc_type}

SUMMARY:
{summary}

Return:
STATUS: PASS / CONDITIONAL / FAIL
MISSING_ITEMS:
- item:
  reason:
REVIEWER_ACTIONS:
- action:
"""

    return call_llm(prompt, "You are a strict regulatory compliance checker.")


# =========================
# COMPLETENESS ASSESSMENT
# =========================
def assess_completeness(text: str, doc_type: str) -> str:
    rules = retrieve_guidelines(
        f"{doc_type} CDSCO SUGAM mandatory forms administrative checklist clinical approval SAE required fields"
    )

    prompt = f"""
You are a strict CDSCO regulatory reviewer.

DOCUMENT TYPE:
{doc_type}

REGULATORY CHECKLIST:
{rules}

DOCUMENT TEXT:
{text}

Check:
1. Administrative sections
2. Mandatory forms
3. Regulatory checklist items
4. Clinical approval application fields
5. SAE fields if applicable
6. Missing fields
7. Inconsistent fields
8. Accuracy issues
9. Weak or unclear sections

Output structured YAML only:

COMPLETENESS_STATUS: PASS / CONDITIONAL / FAIL

MISSING_FIELDS:
- field:
  reason:
  severity:

INCONSISTENCIES:
- issue:
  evidence:
  severity:

ACCURACY_CONCERNS:
- issue:
  evidence:
  severity:

RECOMMENDED_REVIEWER_ACTIONS:
- action:
"""

    return call_llm(prompt, "You are a strict CDSCO completeness auditor.")


# =========================
# SAE SEVERITY
# =========================
SEVERITY_KEYWORDS = {
    "DEATH": ["death", "died", "fatal", "expired", "mortality", "passed away"],
    "DISABILITY": ["disability", "disabled", "permanent impairment", "paralysis", "loss of function", "blindness"],
    "HOSPITALISATION": ["hospitalized", "hospitalised", "admitted", "icu", "inpatient", "emergency admission"],
}


def rule_based_severity(text: str) -> str:
    t = text.lower()

    for label, keywords in SEVERITY_KEYWORDS.items():
        if any(keyword in t for keyword in keywords):
            return label

    return "OTHERS"


def llm_severity_classification(text: str) -> str:
    prompt = f"""
Classify severity into exactly one category:
DEATH, DISABILITY, HOSPITALISATION, OTHERS.

Rules:
- Death/fatal outcome -> DEATH
- Permanent disability/impairment -> DISABILITY
- Admission/ICU/emergency/inpatient -> HOSPITALISATION
- Otherwise -> OTHERS
- Do not guess.

TEXT:
{text}

Output:
SEVERITY:
EVIDENCE:
CONFIDENCE:
"""

    return call_llm(prompt, "You are a strict SAE severity classification engine.")


def classify_severity(text: str, doc_type: str) -> Dict[str, Any]:
    if doc_type != "SAE":
        return {
            "rule_based_severity": "NOT_APPLICABLE",
            "llm_severity_report": "NOT_APPLICABLE",
        }

    return {
        "rule_based_severity": rule_based_severity(text),
        "llm_severity_report": llm_severity_classification(text),
    }


# =========================
# DUPLICATE DETECTION
# =========================
def normalize_for_duplicate(text: str) -> str:
    return " ".join(text.lower().split())


def exact_duplicate_hash(text: str) -> str:
    normalized = normalize_for_duplicate(text)
    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()


def cosine_similarity(text1: str, text2: str) -> float:
    emb = embed_model.encode(
        [text1, text2],
        normalize_embeddings=True,
    )

    emb = np.asarray(emb, dtype="float32")
    return float(np.dot(emb[0], emb[1]))


def detect_duplicates(
    current_result: Dict[str, Any],
    previous_results: List[Dict[str, Any]],
    threshold: float = 0.92,
) -> Dict[str, Any]:

    current_text = current_result.get("anonymized_text", "")

    if not previous_results:
        return {
            "is_duplicate": False,
            "duplicates": [],
        }

    current_hash = exact_duplicate_hash(current_text)
    duplicates = []

    for old in previous_results:
        if not isinstance(old, dict):
            continue

        old_text = old.get("anonymized_text", "")
        old_file = old.get("file", "UNKNOWN_FILE")

        if not old_text:
            continue

        old_hash = exact_duplicate_hash(old_text)

        if current_hash == old_hash:
            duplicates.append({
                "file": old_file,
                "match_type": "EXACT",
                "similarity": 1.0,
            })
        else:
            sim = cosine_similarity(current_text, old_text)

            if sim >= threshold:
                duplicates.append({
                    "file": old_file,
                    "match_type": "NEAR_DUPLICATE",
                    "similarity": round(sim, 3),
                })

    return {
        "is_duplicate": bool(duplicates),
        "duplicates": duplicates,
    }


# =========================
# PRIORITY
# =========================
def assign_priority(
    severity_report: Dict[str, Any],
    duplicate_report: Dict[str, Any],
    completeness_report: str,
) -> Dict[str, str]:

    severity_text = str(severity_report.get("llm_severity_report", "")).upper()

    if "SEVERITY: DEATH" in severity_text:
        priority = "P1_CRITICAL"
        reason = "Death case requires immediate review."
    elif "SEVERITY: DISABILITY" in severity_text:
        priority = "P2_HIGH"
        reason = "Disability case requires high-priority review."
    elif "SEVERITY: HOSPITALISATION" in severity_text:
        priority = "P3_MEDIUM"
        reason = "Hospitalisation case requires timely review."
    else:
        priority = "P4_NORMAL"
        reason = "No critical SAE severity detected."

    if duplicate_report.get("is_duplicate"):
        reason += " Possible duplicate detected."

    if "FAIL" in str(completeness_report).upper():
        reason += " Completeness failure detected."

    return {
        "priority": priority,
        "reason": reason,
    }


# =========================
# VERSION COMPARISON
# =========================
def compare_versions(results: List[Dict[str, Any]]) -> Dict[str, Any]:
    if len(results) < 2:
        return {
            "status": "SKIPPED",
            "reason": "Need at least two documents for version comparison.",
        }

    text1 = results[0].get("anonymized_text", "")
    text2 = results[1].get("anonymized_text", "")

    diff = list(difflib.unified_diff(
        text1.splitlines(),
        text2.splitlines(),
        lineterm="",
    ))

    emb = embed_model.encode([text1, text2], normalize_embeddings=True)
    emb = np.asarray(emb, dtype="float32")
    similarity = float(np.dot(emb[0], emb[1]))

    return {
        "status": "DONE",
        "overall_similarity": round(similarity, 3),
        "diff_preview": "\n".join(diff[:100]),
    }


# =========================
# INSPECTION REPORT GENERATION
# =========================

INSPECTION_TEMPLATE_SECTIONS = {
    "GENERAL": [
        "Name and address of clinical trial site",
        "Date of inspection",
        "Inspection team members",
        "Personnel present during inspection",
        "Investigator address and contact details",
        "Sponsor name and address",
        "Clinical trial NOC holder",
        "Ethics Committee name and address",
        "Protocol title",
        "Protocol number, version/date, amendments",
        "Investigational product",
        "Stage of study",
        "Type of inspection",
    ],

    "LEGAL_ADMINISTRATIVE": [
        "Clinical trial NOC from DCGI",
        "NOC for protocol amendments",
        "Ethics Committee approval",
        "Appendix VII as per Schedule Y",
        "Financial agreement between sponsor, investigator and institution",
        "Liability agreement",
        "Clinical trial insurance",
        "Site initiation date",
        "First subject screening date",
        "First subject ICF signing date",
        "Last patient last follow-up date",
        "SOPs established and documented",
        "Emergency care facilities",
    ],

    "ORGANISATION_PERSONNEL": [
        "Signed and dated CV of investigator and sub-investigators",
        "Medical Council registration",
        "GCP, Schedule Y and protocol training",
        "Duty delegation log",
        "Qualification and training of delegated personnel",
        "List of clinical trials performed in last three years",
        "Investigator not involved in more than three trials at a time",
    ],

    "CONDUCT_OF_TRIAL": [
        "Screening informed consent",
        "Screening and enrolment log",
        "Inclusion and exclusion criteria verification",
        "Clinical examination by investigator",
        "Laboratory evaluation",
        "X-Ray, MRI, ECG, USG or other protocol-required tests",
        "Compliance with clinical trial NOC conditions",
        "ICF contains Appendix V Schedule Y elements",
        "ICF approved by Ethics Committee",
        "Consent obtained before participation",
        "Subject or legal representative signature with date",
        "Impartial witness for illiterate subjects",
        "Investigator signed and dated completed ICF",
        "Re-consenting for ICF changes",
        "Audio-visual consent recording where applicable",
        "Source document completeness and legibility",
        "CRF transcription accuracy",
        "Dropout records",
        "SAE handling SOP",
        "SAE reporting to sponsor, EC and Licensing Authority",
        "Medical care during SAE",
    ],

    "SPONSOR": [
        "Copies of reports submitted to sponsor",
        "CRFs submitted after study completion",
        "Dropouts reported to sponsor",
        "Monitoring method and frequency",
        "Qualified monitor appointed",
        "Onsite monitoring visit log",
        "Monitor visit report with deviations",
        "Sponsor QA audit",
        "Premature termination communication",
    ],

    "INVESTIGATIONAL_PRODUCT": [
        "Correct dose administration",
        "Unauthorized administration or dispensing",
        "Test drug accountability and reconciliation",
        "Storage condition monitoring",
        "Secured access to trial medication",
        "Return or disposal of unused medication",
        "Drug dispensing records",
        "IP reconciliation records",
        "Temperature logs",
        "Appropriate IP labelling",
    ],

    "ETHICS_COMMITTEE": [
        "EC name and address verification",
        "EC registration status",
        "EC approval letter completeness",
        "EC meeting minutes",
        "EC onsite monitoring",
        "Conflict of interest handling",
        "Communication between investigator and EC",
        "EC functioning according to registration conditions",
    ],

    "PATHOLOGY_LAB": [
        "Clinical laboratory name and address",
        "Financial and confidentiality agreement with laboratory",
        "Lab accreditation and facility adequacy",
        "Sample preparation, handling and transport SOP",
    ],

    "QUALITY_ASSURANCE": [
        "Site-specific and trial-specific SOPs",
        "SOP preparation, review, authorization and revision",
        "SOPs for consent, AV recording, SAE, EC/Sponsor/CDSCO communication",
        "SOPs for IP handling and sample handling",
        "SOPs for storage cabinets, refrigerators and freezers",
        "Personnel job description, qualification and training records",
        "Activities compliant with duty delegation",
        "Staff training records",
        "Vaccine spillage SOP if applicable",
    ],

    "RECORD_KEEPING_DATA_HANDLING": [
        "Adequate document retention space",
        "Documents maintained for specified period",
        "Protection from accidental or premature destruction",
        "Controlled archival access",
        "Data management SOP",
        "Corrections carry date and initials",
        "Electronic data processing by authorized person",
        "Authorized persons list",
        "Audit trail for changes and deletions",
        "Hardware and software validation",
    ],
}


def generate_inspection_report_from_observations(observation_text: str) -> str:
    """
    Converts unstructured / handwritten inspection observations into a formal
    CDSCO-style GCP inspection report.
    """

    template_text = ""

    for section, items in INSPECTION_TEMPLATE_SECTIONS.items():
        template_text += f"\n\nSECTION: {section}\n"
        for i, item in enumerate(items, start=1):
            template_text += f"{i}. {item}\n"

    prompt = f"""
You are a senior CDSCO GCP inspection officer.

TASK:
Convert the unstructured site inspection observations into a formal inspection report
conforming to the CDSCO GCP inspection checklist format.

Use this checklist structure:

{template_text}

INSPECTION OBSERVATIONS:
{observation_text}

For every checklist item, classify status as:
YES / NO / NA / NOT_RECORDED

Also identify:
- Critical non-compliance
- Major non-compliance
- Minor observation
- Missing evidence
- Required authenticated copies/exhibits
- Recommended corrective and preventive actions

OUTPUT FORMAT:

GCP_INSPECTION_REPORT:

GENERAL_INFORMATION:
  Clinical trial site:
  Date of inspection:
  Inspection team:
  Personnel present:
  Investigator:
  Sponsor:
  NOC holder:
  Ethics Committee:
  Protocol title:
  Protocol number/version:
  Investigational product:
  Stage of study:
  Type of inspection:

INSPECTION_FINDINGS:
  - section:
    checklist_item:
    status: YES / NO / NA / NOT_RECORDED
    observation:
    evidence:
    severity: CRITICAL / MAJOR / MINOR / NONE
    recommendation:
    required_exhibit:

NON_COMPLIANCE_SUMMARY:
  Critical:
    - issue:
      evidence:
      action_required:
  Major:
    - issue:
      evidence:
      action_required:
  Minor:
    - issue:
      evidence:
      action_required:

MISSING_INFORMATION:
  - item:
    reason:

RECOMMENDED_CAPA:
  - finding:
    corrective_action:
    preventive_action:
    responsible_party:

FINAL_INSPECTION_CONCLUSION:
  Overall_status: SATISFACTORY / CONDITIONAL / UNSATISFACTORY
  Reason:
  Immediate_actions_required:

Return only structured YAML.
"""

    return call_llm(
        prompt,
        system="You are a strict CDSCO GCP inspection report generation engine."
    )

# =========================
# MAIN PIPELINE
# =========================
def process_pdf(
    file_path: str,
    previous_results: List[Dict[str, Any]] = None,
) -> Dict[str, Any]:

    if previous_results is None:
        previous_results = []

    print(f"\nProcessing: {file_path}")

    text = extract_text_from_pdf(file_path)

    if not text.strip():
        raise ValueError(f"No extractable text found in PDF: {file_path}")

    spans = merge_spans(rule_detect(text) + nlp_detect(text))
    anon_text, token_map = anonymize(text, spans)

    print("Initial anonymization done.")

    doc_type = classify_doc_type(anon_text)
    print(f"Document classified as: {doc_type}")

    refined_text = refine_with_llm(anon_text)
    print("LLM refinement done.")

    privacy = compliance_check(refined_text)
    print(f"Privacy compliance: {privacy['status']}")

    summary = summarize(refined_text, doc_type)
    print("Summarization done.")

    regulatory = regulatory_compliance(summary, doc_type)
    print("Regulatory compliance done.")

    completeness = assess_completeness(refined_text, doc_type)
    print("Completeness assessment done.")

    inspection_report = generate_inspection_report_from_observations(refined_text)
    print("Inspection report generation done.")

    severity = classify_severity(refined_text, doc_type)

    current_result = {
        "file": str(file_path),
        "anonymized_text": refined_text,
    }

    duplicate_report = detect_duplicates(
        current_result=current_result,
        previous_results=previous_results,
    )

    priority = assign_priority(
        severity_report=severity,
        duplicate_report=duplicate_report,
        completeness_report=completeness,
    )
    print(refined_text)

    print(inspection_report)

    return {
        "file": str(file_path),
        "raw_text_length": len(text),
        "token_map": token_map,
        "anonymized_text": refined_text,
        "privacy": privacy,
        "type": doc_type,
        "summary": summary,
        "regulatory": regulatory,
        "completeness": completeness,
        "inspection_report": inspection_report,
        "severity": severity,
        "duplicates": duplicate_report,
        "priority": priority,
    }


# =========================
# PROCESS MULTIPLE PDFS
# =========================
def process_multiple_pdfs(file_list: List[str]) -> List[Dict[str, Any]]:
    results = []

    for file_path in file_list:
        try:
            result = process_pdf(
                file_path=file_path,
                previous_results=results,
            )

            results.append(result)

            print("\nCompleted:", file_path)
            print("Privacy:", result["privacy"]["status"])
            print("Priority:", result["priority"]["priority"])

        except Exception as e:
            print(f"Failed: {file_path}")
            print("Error:", e)

            results.append({
                "file": str(file_path),
                "error": str(e),
            })

    return results


# =========================
# TEST
# =========================
if __name__ == "__main__":
    files = [
        "good_application_1.pdf",
        # "good_application_2.pdf",
    ]

    results = process_multiple_pdfs(files)

    version_report = compare_versions([
        r for r in results
        if "anonymized_text" in r
    ])

    print("\n====================")
    print("FINAL RESULTS")
    print("====================")

    for r in results:
        print("\n---")
        print("File:", r.get("file"))
        print("Type:", r.get("type"))
        print("Privacy:", r.get("privacy"))
        print("Priority:", r.get("priority"))
        print("Error:", r.get("error"))

    print("\nVERSION REPORT:")
    print(version_report)

In [ ]:
# =========================
# GRADIO UI - NO MULTIPROCESSING
# =========================
import gradio as gr
import json
from datetime import datetime


def safe_json(data):
    try:
        return json.dumps(data, indent=2, ensure_ascii=False)
    except Exception:
        return str(data)


def run_single_file(file_path, previous_results):
    result = process_pdf(
        file_path=file_path,
        previous_results=previous_results
    )
    return result


def run_gradio_pipeline(uploaded_files):
    if not uploaded_files:
        return (
            "Please upload at least one PDF file.",
            "",
            "",
            "",
            "",
            "",
            "",
            "",
            "",
            {},
            None
        )

    results = []
    overview_lines = []

    for file_path in uploaded_files:
        try:
            result = run_single_file(
                file_path=file_path,
                previous_results=results
            )

            results.append(result)

            overview_lines.append(
                f"""
FILE: {result.get("file")}
TYPE: {result.get("type")}
PRIVACY STATUS: {result.get("privacy", {}).get("status")}
PRIORITY: {result.get("priority", {}).get("priority")}
REASON: {result.get("priority", {}).get("reason")}
"""
            )

        except Exception as e:
            error_result = {
                "file": str(file_path),
                "error": str(e)
            }
            results.append(error_result)

            overview_lines.append(
                f"""
FILE: {file_path}
STATUS: FAILED
ERROR: {str(e)}
"""
            )

    valid_results = [
        r for r in results
        if "anonymized_text" in r
    ]

    version_report = compare_versions(valid_results)

    if not valid_results:
        output_json_path = save_results_to_json(results, version_report)

        return (
            "\n".join(overview_lines),
            "",
            "",
            "",
            "",
            "",
            "",
            "",
            safe_json(results),
            version_report,
            output_json_path
        )

    first = valid_results[0]

    output_json_path = save_results_to_json(results, version_report)

    return (
        "\n".join(overview_lines),
        first.get("anonymized_text", ""),
        first.get("summary", ""),
        first.get("regulatory", ""),
        first.get("completeness", ""),
        first.get("inspection_report", ""),
        safe_json(first.get("severity", {})),
        safe_json(first.get("duplicates", {})),
        safe_json(results),
        version_report,
        output_json_path
    )


def save_results_to_json(results, version_report):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_path = f"cdsco_review_output_{timestamp}.json"

    output_data = {
        "generated_at": timestamp,
        "results": results,
        "version_report": version_report
    }

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(output_data, f, indent=2, ensure_ascii=False)

    return output_path


with gr.Blocks(title="CDSCO Regulatory Review Automation") as demo:

    gr.Markdown("""
    # CDSCO Regulatory Review Automation

    Upload one or more PDF files.

    This UI shows:
    - PII/PHI anonymization
    - Privacy compliance
    - Regulatory summary
    - Completeness assessment
    - CDSCO GCP inspection report
    - SAE severity
    - Duplicate detection
    - Version comparison
    """)

    uploaded_files = gr.File(
        label="Upload PDF Files",
        file_types=[".pdf"],
        file_count="multiple",
        type="filepath"
    )

    run_button = gr.Button("Run Pipeline")

    with gr.Tab("Overview"):
        overview_output = gr.Textbox(
            label="Processing Overview",
            lines=18
        )

    with gr.Tab("Anonymized Text"):
        anonymized_output = gr.Textbox(
            label="Anonymized Text",
            lines=25
        )

    with gr.Tab("Summary"):
        summary_output = gr.Textbox(
            label="Document Summary",
            lines=25
        )

    with gr.Tab("Regulatory Compliance"):
        regulatory_output = gr.Textbox(
            label="Regulatory Compliance",
            lines=25
        )

    with gr.Tab("Completeness"):
        completeness_output = gr.Textbox(
            label="Completeness Assessment",
            lines=25
        )

    with gr.Tab("Inspection Report"):
        inspection_output = gr.Textbox(
            label="CDSCO GCP Inspection Report",
            lines=30
        )

    with gr.Tab("SAE Severity"):
        severity_output = gr.Code(
            label="Severity Output",
            language="json"
        )

    with gr.Tab("Duplicate Detection"):
        duplicate_output = gr.Code(
            label="Duplicate Detection Output",
            language="json"
        )

    with gr.Tab("Full JSON"):
        full_json_output = gr.Code(
            label="Complete Pipeline JSON",
            language="json"
        )

    with gr.Tab("Version Comparison"):
        version_output = gr.JSON(
            label="Version Comparison"
        )

    with gr.Tab("Download Output"):
        download_output = gr.File(
            label="Download JSON Output"
        )

    run_button.click(
        fn=run_gradio_pipeline,
        inputs=[uploaded_files],
        outputs=[
            overview_output,
            anonymized_output,
            summary_output,
            regulatory_output,
            completeness_output,
            inspection_output,
            severity_output,
            duplicate_output,
            full_json_output,
            version_output,
            download_output
        ]
    )


if __name__ == "__main__":
    demo.launch()